# InstrumentTrainer — trening w Colab

1. Runtime → **GPU** (opcjonalnie, przyspiesza trening)
2. Ustaw token Kaggle (Secrets: `KAGGLE_API_TOKEN` lub upload `access_token` do `~/.kaggle/`)
3. Uruchom komórki po kolei

In [ ]:
# Klonuj repo (albo wgraj projekt ręcznie)
!git clone https://github.com/wiktorlee/PianoNotesNeuralNetwork.git
%cd PianoNotesNeuralNetwork

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
from google.colab import userdata

# Colab Secrets NIE trafiają do os.environ — użyj userdata.get("NAZWA_SECRETA")
# U Ciebie secret nazywa się "KAGGLE" (włącz Notebook access)
for secret_name in ("KAGGLE", "KAGGLE_API_TOKEN"):
    try:
        token = userdata.get(secret_name).strip()
        break
    except userdata.SecretNotFoundError:
        token = None
else:
    raise ValueError(
        "Brak secretu. Colab: ikona klucza → Name: KAGGLE → włącz Notebook access."
    )

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / "access_token").write_text(token)
print(f"Token OK (secret: {secret_name}, długość {len(token)})")

In [ ]:
!kaggle datasets download -d riccardosimionato/pianorecordingssinglenotes -p data_raw --unzip

In [ ]:
!python scripts/preprocess.py

In [ ]:
!python scripts/train.py --mixed-precision --no-plots --epochs 30

In [ ]:
from google.colab import files
for p in ["models/piano_cnn.tflite", "models/training_history.png", "models/confusion_matrix.png"]:
    if Path(p).exists():
        files.download(p)